# 06. Regressão Linear — Parte 1
## Beta CAPM: se o mercado cair 10%, quanto cai ITUB3?

---

## Cenário

Você é analista de risco de mercado. O gestor sênior te chama antes do comitê de alocação:

> *"O fundo tem uma posição relevante em ITUB3. Se o mercado cair 10% amanhã, quanto essa ação deve cair? Ela amplifica o mercado ou atenua?"*

Essa pergunta tem uma resposta quantitativa. O **Beta do CAPM**, estimado por regressão linear simples, é a ferramenta para encontrá-la.

---

## Estrutura do notebook

| Seção | Conteúdo | Entregável |
|---|---|---|
| 1. Dados | yfinance → retornos mensais ITUB3 e Ibovespa | — |
| 2. EDA | Scatter e correlação | — |
| 3. Hipótese | Por que o Ibovespa explica ITUB3? | **⭐ Entregável** |
| 4. Modelo | Regressão com sklearn, Beta, Alfa, R² | **⭐ Entregável** |
| 5. Resíduos | Diagnóstico gráfico | **⭐ Entregável** |
| 6. Conclusão | Mensagem para o gestor + limitações | — |

---

## 1. Dados

In [ ]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
})

In [ ]:
# 10 anos de dados: múltiplos ciclos de mercado — essencial para um Beta robusto
START, END = '2015-01-01', '2024-12-31'

raw_itub = yf.download('ITUB3.SA', start=START, end=END, auto_adjust=True, progress=False)
raw_ibov = yf.download('^BVSP',    start=START, end=END, auto_adjust=True, progress=False)

preco_itub = raw_itub['Close'].squeeze().rename('ITUB3')
preco_ibov = raw_ibov['Close'].squeeze().rename('IBOV')

# Resampling mensal (último pregão) → retornos logarítmicos
# r_t = ln(P_t / P_{t-1})  — aditivos no tempo e mais simétricos que retornos simples
df = pd.DataFrame({
    'r_itub': np.log(preco_itub.resample('ME').last() / preco_itub.resample('ME').last().shift(1)),
    'r_ibov': np.log(preco_ibov.resample('ME').last() / preco_ibov.resample('ME').last().shift(1)),
}).dropna()

print(f'Período: {df.index.min().strftime("%b/%Y")} a {df.index.max().strftime("%b/%Y")} | {len(df)} observações mensais')
print()
(df * 100).describe().round(2)

**Leituras dos dados:**

- **Volatilidade:** O desvio padrão de ITUB3 é maior que o do Ibovespa — primeiro indício de que a ação amplifica movimentos do mercado (β > 1 candidato).
- **Mínimo:** O pior mês de ambas as séries coincide com março/2020 (crash COVID). Quando o mercado inteiro cai junto, não há diversificação possível — isso é **risco sistemático** em ação.
- **10 anos de dados** cobrem bull market (2016–2019), crash e recuperação COVID (2020), ciclo de alta de juros (2021–2023) — sem isso, o Beta seria estimado em um único regime e seria pouco confiável.

---

## 2. Análise Exploratória

Antes de qualquer modelo: **ver os dados**. A pergunta central é se a relação entre ITUB3 e o Ibovespa parece linear — porque é isso que a regressão OLS assume.

In [ ]:
r_pearson, p_valor = stats.pearsonr(df['r_ibov'], df['r_itub'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Painel esquerdo: séries temporais ---
ax = axes[0]
ax.plot(df.index, df['r_itub'] * 100, alpha=0.7, linewidth=1.2, label='ITUB3', color='#2166ac')
ax.plot(df.index, df['r_ibov'] * 100, alpha=0.7, linewidth=1.2, label='Ibovespa', color='#d6604d')
ax.axhline(0, color='black', linewidth=0.7, linestyle=':')
ax.axvline(pd.Timestamp('2020-03-01'), color='crimson', linestyle='--', linewidth=1.2, alpha=0.8)
ax.text(pd.Timestamp('2020-04-01'), 14, 'Crash\nCOVID', fontsize=8.5, color='crimson')
ax.set_ylabel('Retorno mensal (%)')
ax.set_title('Retornos Mensais: ITUB3 e Ibovespa')
ax.legend(fontsize=9)

# --- Painel direito: scatter com IC 95% ---
ax = axes[1]
sns.regplot(
    data=df * 100, x='r_ibov', y='r_itub',
    scatter_kws={'alpha': 0.55, 's': 40, 'color': '#4393c3'},
    line_kws={'color': '#d6604d', 'linewidth': 2},
    ci=95, ax=ax
)
crash = df.loc['2020-03']
ax.scatter(crash['r_ibov'] * 100, crash['r_itub'] * 100,
           color='crimson', s=90, zorder=5, label='Mar/2020')
ax.annotate('Mar/2020', (crash['r_ibov'] * 100, crash['r_itub'] * 100),
            xytext=(6, -10), textcoords='offset points', fontsize=8.5, color='crimson')
ax.axhline(0, color='gray', linewidth=0.7, linestyle=':')
ax.axvline(0, color='gray', linewidth=0.7, linestyle=':')
ax.set_xlabel('Retorno mensal Ibovespa (%)')
ax.set_ylabel('Retorno mensal ITUB3 (%)')
ax.set_title(f'Scatter: r = {r_pearson:.3f} | p = {p_valor:.2e}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Correlação de Pearson: r = {r_pearson:.4f} | p-valor = {p_valor:.4e}')

**O que o scatter nos conta antes de qualquer modelo:**

1. **Correlação forte e significante** — o p-valor indica que a probabilidade de observar essa correlação por acaso é essencialmente zero. Há um fator comum movendo os dois ativos.

2. **Inclinação visual acima de 45°** — a reta parece mais inclinada do que a diagonal de 45° (que representaria β = 1). Isso sugere que ITUB3 amplifica os movimentos do Ibovespa — candidato a β > 1.

3. **Mar/2020 (crash COVID)** — ponto extremo de alta influência. Ele está longe da nuvem central e por isso exerce pressão desproporcional sobre a inclinação estimada. Faremos o diagnóstico de sua influência na análise de resíduos.

---

## 3. Hipótese ⭐

### 3.1 Formulação do modelo

**Variável dependente (Y):** Retorno logarítmico mensal de ITUB3 — ação ordinária do Itaú Unibanco.

**Variável explicativa (X):** Retorno logarítmico mensal do Ibovespa — proxy do portfólio de mercado brasileiro.

**Equação da Linha de Mercado do Título (SML):**

$$r_{\text{ITUB3},\,t} = \alpha + \beta \cdot r_{\text{IBOV},\,t} + \varepsilon_t$$

| Parâmetro | Nome | Pergunta que responde |
|---|---|---|
| $\alpha$ | Alfa de Jensen | *ITUB3 entrega retorno além do compensado pelo risco?* |
| $\beta$ | Beta do CAPM | *Quanto ITUB3 se move para cada 1% de movimento do Ibovespa?* |
| $\varepsilon_t$ | Risco idiossincrático | *O que o mercado não explica — eventos específicos do banco* |

---

### 3.2 Por que o Ibovespa explica ITUB3?

No **CAPM** (Sharpe, 1964), todo ativo tem seu retorno decomposto em dois componentes:

$$\underbrace{\beta \cdot r_{\text{IBOV}}}_{{\text{risco sistemático}}} + \underbrace{\varepsilon_t}_{\text{risco idiossincrático}}$$

O **risco sistemático** é o risco que afeta todos os ativos ao mesmo tempo — guerras, crises financeiras, choques macroeconômicos. Não pode ser eliminado por diversificação. O Ibovespa captura esse fator comum.

O **risco idiossincrático** é específico do banco — fraude, mudança de gestão, resultado acima do esperado. Pode ser eliminado com diversificação.

**Por que o Ibovespa é o X natural:**

1. **É o benchmark do mercado brasileiro** — composto pelas ações mais líquidas da B3, captura o comportamento médio do mercado doméstico.

2. **Canal de transmissão direto para bancos:**
   ```
   Mercado cai (Ibov ↓)
     → Apetite por risco ↓ → venda de ativos de risco (bancos incluídos)
     → Expectativa de recessão ↑ → inadimplência esperada ↑
       → Valuation do banco ↓ (provisão maior, margem financeira menor)
   ```

3. **Bancos são pró-cíclicos por natureza** — seus lucros crescem na expansão (crédito aquecido, spread alto) e caem na recessão (inadimplência, queda de volume). Espera-se **β > 1** estruturalmente.

4. **Variável pública e de alta qualidade** — série histórica longa, sem gaps relevantes.

---

### 3.3 Hipótese e direção esperada

**Hipótese:** ITUB3 tem Beta **maior que 1** — ativo pró-cíclico que amplifica os movimentos do Ibovespa.

**Direção esperada:** $\hat{\beta} > 1$

| Evento | Ibovespa | ITUB3 (esperado) | Evidência |
|---|---|---|---|
| Crash COVID (mar/2020) | −29% | Queda > 29% | Bancos: inadimplência ↑, crédito cai |
| Recuperação (2020–2021) | +70% | Alta > 70% | Recuperação acelerada |
| Setor bancário estruturalmente | Referência | Amplifica | Pró-ciclico por natureza |

---

### 3.4 O que o $R^2$ vai medir

No CAPM, o $R^2$ tem interpretação de risco:

$$R^2 = \frac{\text{Risco Sistemático}}{\text{Risco Total}} = 1 - \frac{\text{Risco Idiossincrático}}{\text{Risco Total}}$$

Um $R^2 = 0{,}70$ significa que 70% da volatilidade de ITUB3 é sistemática (não diversificável) — e 30% é específica do banco.

---

## 4. Modelo — Regressão com sklearn ⭐

In [ ]:
X = df['r_ibov'].values.reshape(-1, 1)  # variável explicativa: retorno do Ibovespa
y = df['r_itub'].values                 # variável dependente: retorno de ITUB3

modelo = LinearRegression()
modelo.fit(X, y)
y_pred = modelo.predict(X)
residuos = y - y_pred

alfa  = modelo.intercept_
beta  = modelo.coef_[0]
r2    = r2_score(y, y_pred)
rmse  = np.sqrt(np.mean(residuos ** 2))

print('=== Modelo estimado ===')
print(f'  r_ITUB3 = {alfa*100:.4f}% + {beta:.4f} × r_IBOV')
print()
print(f'  Alfa de Jensen  α = {alfa*100:.4f}% ao mês  ({((1+alfa)**12-1)*100:+.2f}% a.a.)')
print(f'  Beta CAPM       β = {beta:.4f}')
print()
print(f'  R²   = {r2:.4f}  ({r2*100:.1f}% da volatilidade de ITUB3 é sistemática)')
print(f'  RMSE = {rmse*100:.2f} p.p.  (erro médio mensal do modelo)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(X * 100, y * 100, alpha=0.55, s=40, color='#4393c3',
           label='Retornos mensais observados', zorder=3)

# Reta OLS: Linha de Mercado do Título (SML) empírica
x_linha = np.linspace(X.min(), X.max(), 200)
ax.plot(x_linha * 100, (alfa + beta * x_linha) * 100,
        color='#d6604d', linewidth=2.2, label='SML empírica (OLS)', zorder=4)

# Referência β = 1: reta de 45° — acima = ativo amplifica; abaixo = ativo atenua
ax.plot(x_linha * 100, x_linha * 100,
        color='gray', linewidth=1.2, linestyle='--', alpha=0.7, label='β = 1 (referência)')

ax.axhline(0, color='black', linewidth=0.6, linestyle=':')
ax.axvline(0, color='black', linewidth=0.6, linestyle=':')

equacao = f'r_ITUB3 = {alfa*100:.3f}% + {beta:.4f} × r_IBOV\nR² = {r2:.4f}'
ax.text(0.05, 0.92, equacao, transform=ax.transAxes, fontsize=10.5,
        va='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

ax.set_xlabel('Retorno mensal Ibovespa (%)')
ax.set_ylabel('Retorno mensal ITUB3 (%)')
ax.set_title('Linha de Mercado do Título (SML) — Beta CAPM Estimado')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Interpretação dos coeficientes

**Beta (β):**

> *"Para cada 1% de retorno do Ibovespa, ITUB3 sobe/cai β%, em média."*

| Valor de β | Classificação | Comportamento |
|---|---|---|
| β < 1 | Defensivo | Atenua os movimentos do mercado |
| β = 1 | Neutro | Replica o mercado |
| β > 1 | Agressivo | Amplifica os movimentos do mercado |

**Alfa de Jensen (α):**

> *"Retorno médio de ITUB3 acima (ou abaixo) do previsto pelo CAPM."*

- α > 0: o banco gerou retorno além do compensado pelo risco — *outperformance* ajustada
- α ≈ 0: alinhado com a hipótese de eficiência de mercado — o ativo é compensado exatamente pelo risco assumido
- α < 0: destruição de valor ajustada ao risco

> **Atenção:** o Alfa só é relevante se o intervalo de confiança **não incluir zero**. Em janelas longas, raramente é estatisticamente significante.

**R² e decomposição do risco:**

$$R^2 = \frac{\text{Risco Sistemático}}{\text{Risco Total}} \quad\Rightarrow\quad \text{Risco Idiossincrático} = (1 - R^2) \times \text{Risco Total}$$

No CAPM, somente o risco sistemático é **compensado** — o risco idiossincrático pode ser eliminado gratuitamente por diversificação, portanto o mercado não paga prêmio por ele. Um gestor que carrega alto risco idiossincrático sem diversificar está assumindo risco sem remuneração.

In [ ]:
print('=== Aplicação prática: respondendo ao gestor ===')
print()

# Cenário 1: queda de 10% do Ibovespa (a pergunta original do comitê)
queda_ibov = -0.10
retorno_previsto = alfa + beta * queda_ibov
print(f'Cenário: Ibovespa cai 10%')
print(f'  Retorno previsto de ITUB3: {retorno_previsto*100:+.2f}%')
print(f'  ITUB3 {"amplifica" if abs(retorno_previsto) > abs(queda_ibov) else "atenua"} a queda do mercado')
print()

# Decomposição da volatilidade
vol_itub = y.std() * np.sqrt(12) * 100
vol_ibov = df['r_ibov'].values.std() * np.sqrt(12) * 100
vol_idio = residuos.std() * np.sqrt(12) * 100

print(f'=== Decomposição do risco (volatilidade anualizada) ===')
print(f'  Risco total de ITUB3:      {vol_itub:.1f}% a.a.')
print(f'  Ibovespa (mercado):        {vol_ibov:.1f}% a.a.')
print(f'  Risco sistemático (β×σ_m): {beta * vol_ibov:.1f}% a.a.  ({r2*100:.0f}% do total)')
print(f'  Risco idiossincrático:     {vol_idio:.1f}% a.a.  ({(1-r2)*100:.0f}% do total)')
print()
print(f'  RMSE mensal: {rmse*100:.2f} p.p. — erro médio do modelo em um mês típico')

---

## 5. Análise dos Resíduos ⭐

Os resíduos desta regressão têm um nome próprio em finanças: **retornos anormais** (*abnormal returns*). São os movimentos de ITUB3 que o mercado **não** explica — eventos específicos do banco.

O gráfico de resíduos vs. valores preditos é o diagnóstico central do OLS:

| Padrão visual | Significado | Consequência |
|---|---|---|
| Dispersão aleatória em torno de zero | Modelo bem especificado | OLS é BLUE |
| Curvatura na LOWESS | Não-linearidade | β captura inclinação média de relação não-linear |
| Dispersão crescente | Heteroscedasticidade | Erros-padrão incorretos → ICs inválidos |
| Clusters sistemáticos | Variável omitida / quebra estrutural | Coeficientes viesados |

In [ ]:
smooth = lowess(residuos, y_pred, frac=0.4, return_sorted=True)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(y_pred * 100, residuos * 100,
           alpha=0.5, s=35, color='#4393c3', zorder=3, label='Retornos anormais (resíduos)')
ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
ax.plot(smooth[:, 0] * 100, smooth[:, 1] * 100,
        color='#d6604d', linewidth=2.2, label='LOWESS (tendência local)', zorder=4)

# Destaca pontos do crash COVID (mar-abr/2020)
mask_covid = df.index.year == 2020
ax.scatter(y_pred[mask_covid] * 100, residuos[mask_covid] * 100,
           color='crimson', s=65, zorder=5, edgecolors='darkred', linewidth=0.8,
           label='2020 (COVID)')

# Destaca ciclo de alta da Selic (2021-2022)
mask_selic = df.index.year.isin([2021, 2022])
ax.scatter(y_pred[mask_selic] * 100, residuos[mask_selic] * 100,
           color='darkorange', s=55, zorder=5, edgecolors='saddlebrown', linewidth=0.8,
           label='2021–2022 (ciclo Selic)')

ax.set_xlabel('Retorno predito ITUB3 (%)')
ax.set_ylabel('Resíduo = retorno anormal (%)')
ax.set_title('Diagnóstico: Resíduos vs. Valores Preditos (Retornos Anormais)')
ax.legend(fontsize=9)

ax.text(0.98, 0.97,
        'LOWESS ≈ horizontal → linearidade razoável\n'
        'Dispersão maior nas extremidades → possível heteroscedasticidade condicional (ARCH)',
        transform=ax.transAxes, fontsize=8.5, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

plt.tight_layout()
plt.show()

**Leitura do gráfico de resíduos:**

**Curva LOWESS:** Se aproximadamente horizontal em torno de zero, a relação linear é uma aproximação razoável. Pequenos desvios nas extremidades são esperados — a relação entre retornos de ativos raramente é perfeitamente linear para todos os níveis de mercado.

**Pontos de março/2020 (vermelho):** O crash COVID é o ponto de maior *alavancagem* (*high-leverage point*) da série. Ele está longe da nuvem central e exerce influência desproporcional sobre a inclinação estimada. O Beta calculado com esse ponto pode ser diferente do Beta sem ele — o diagnóstico robusto passaria pela **Distância de Cook** para quantificar essa influência.

**Ciclo Selic 2021–2022 (laranja):** Em 2021–2022, o Banco Central elevou a Selic de 2% para 13,75% em 18 meses. Bancos como o Itaú têm margem financeira ligada ao spread de juros — a alta da Selic é parcialmente positiva para eles (spreads maiores), enquanto prejudica outros setores. Isso cria uma dinâmica diferente do restante do Ibovespa e pode gerar resíduos sistemáticos no período.

**Heteroscedasticidade condicional (ARCH):** A dispersão dos resíduos tende a ser maior em períodos de alta volatilidade de mercado (crise) do que em períodos calmos. Esse é o fundamento empírico dos modelos GARCH — que modelam explicitamente a variância condicional dos retornos ao longo do tempo. Para gestão de risco em instituições reguladas, o OLS simples sem modelagem de ARCH subestima sistematicamente o risco em períodos de estresse.

---

## 6. Conclusão

### O que levar para o gestor

In [ ]:
print('=== Sumário executivo para o comitê de alocação ===')
print()
print(f'Modelo: r_ITUB3 = {alfa*100:.4f}% + {beta:.4f} × r_IBOV')
print(f'Período: 2015-2024 | {len(df)} observações mensais')
print()

classificacao = 'AGRESSIVO (amplifica o mercado)' if beta > 1 else 'DEFENSIVO (atenua o mercado)' if beta < 1 else 'NEUTRO (replica o mercado)'
print(f'1. Beta (β = {beta:.4f}): ativo {classificacao}')
print(f'   → Se o Ibovespa cair 10%, ITUB3 deve cair {abs(alfa + beta * (-0.10))*100:.1f}% em média')
print()
print(f'2. Alfa de Jensen (α = {alfa*100:.4f}% a.m. | {((1+alfa)**12-1)*100:+.2f}% a.a.):')
print(f'   → Retorno acima do CAPM; avaliar significância estatística antes de concluir')
print()
print(f'3. R² = {r2:.4f}: {r2*100:.0f}% do risco de ITUB3 é sistemático (mercado)')
print(f'   → {(1-r2)*100:.0f}% é idiossincrático (banco) — pode ser diluído por diversificação')
print()
print(f'4. RMSE = {rmse*100:.2f} p.p.: incerteza mensal residual do modelo')
print()
print('=== Limitações a comunicar ===')
print(f'  • Beta estimado é uma média histórica — varia com o ciclo econômico')
print(f'  • Crash COVID (mar/2020) exerce influência alta sobre β — ponto de alavancagem')
print(f'  • Resíduos com heteroscedasticidade → erros-padrão do OLS subestimados')
print(f'  • Um único fator: Fama-French (3 ou 5 fatores) capturaria mais variação')

### Síntese dos diagnósticos

| Problema | Sinal diagnóstico | Consequência prática |
|---|---|---|
| **Heteroscedasticidade condicional** | Dispersão dos resíduos maior em crises | Erros-padrão do OLS são incorretos; usar HC3 ou GARCH |
| **Beta não-estacionário** | Varia com o ciclo (bull vs. bear) | Hedge estático com β histórico pode ser ineficiente |
| **Um único fator** | 1 − R² = risco idiossincrático não capturado | Modelos multi-fator (Fama-French) explicam mais |
| **Ponto de alta influência** | Mar/2020 domina o scatter | β sensível à inclusão/exclusão de crises |

---

### Próximos passos — Parte 2

1. **Intervalo de confiança do Beta** — testar se β é estatisticamente diferente de 1 (não apenas de 0)
2. **Beta Rolling** — janela de 24 meses para medir instabilidade do parâmetro ao longo do ciclo
3. **Erros-padrão robustos (HC3)** — corrigir inferência na presença de heteroscedasticidade
4. **Modelo Fama-French 3 fatores** — adicionar SMB (tamanho) e HML (valor) como regressores
5. **Beta assimétrico** — estimar β separado para mercados em alta e em baixa

---

> *"Todos os modelos estão errados, mas alguns são úteis."* — George E. P. Box
>
> O CAPM está errado por construção — o mercado não é perfeitamente eficiente, os investidores não são racionais, e um único fator não captura toda a variação de retornos. Mas ele **é útil**: força a quantificação do risco sistemático, estabelece uma baseline de retorno esperado, e é o ponto de partida de qualquer análise de risco de mercado sério. O que separa o analista que usa o modelo com inteligência do que o usa com ingenuidade é exatamente o que fizemos aqui: entender os pressupostos, executar os diagnósticos, e comunicar as limitações antes de tomar decisões.